In [5]:
import pandas as pd
import seaborn as sns
import numpy as np

data = pd.read_csv("gasoline.csv")

# Split into two groups, one with octane greater than 87.5 and one with less than 86

grp_greater = data[data["octane"] > 87.5]
grp_greater["label"] = np.ones(grp_greater.shape[0])
grp_less = data[data["octane"] < 86]
grp_less["label"] = -1*np.ones(grp_less.shape[0])


X = pd.concat([grp_greater, grp_less], ignore_index=True)
y = X["label"]
X = X.drop(columns=["label", "octane"])

weights = np.ones(X.shape[1])
bias = 0.0

def perceptron(y, x, weights, bias):
    p = y * (np.dot(x, weights) + bias)
    return 1 if  p < 0 else 0

def update(w, b, a, t, idx):
    x = X.iloc[idx].to_numpy()
    Y = y.iloc[idx]

    error = perceptron(Y, x, w, b)

    w = w + a[t] * Y * x * error
    b = b + a[t] * Y * error

    return w, b

iters = 100

a = [100 / t**0.6 for t in range(1, iters + 1)]

incorrectly_classified = np.array(
    [perceptron(y[i], X.iloc[i], weights, bias)
     for i in range(len(X))])



for it in range(iters):

    indexes = np.where(incorrectly_classified == 1)[0]

    if len(indexes) == 0:
        print(f"Converged after {it} iterations!")
        break

    idx = np.random.choice(indexes)

    weights, bias = update(weights, bias, a, it, idx)

    incorrectly_classified = np.array([
        perceptron(y.iloc[i], X.iloc[i].to_numpy(), weights, bias)
        for i in range(len(X))
    ])


C:\Users\thoma\AppData\Local\Temp\ipykernel_14936\904019332.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grp_greater["label"] = np.ones(grp_greater.shape[0])
C:\Users\thoma\AppData\Local\Temp\ipykernel_14936\904019332.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grp_less["label"] = -1*np.ones(grp_less.shape[0])


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("gasoline.csv")

# Prepare data
grp_less = data[data["octane"] < 86].copy()
grp_greater = data[data["octane"] > 87.5].copy()

grp_less["label"] = -1
grp_greater["label"] = 1

combined = pd.concat([grp_less, grp_greater], ignore_index=True)

y = combined["label"].to_numpy(dtype=float)
X = combined.drop(columns=["label", "octane"]).to_numpy(dtype=float)

# Initialization
rng = np.random.default_rng(42)

n, k = X.shape
weights = rng.normal(0, 1e-6, k)
bias = 0.0

iters = 1000
a = [0.5 / t for t in range(1, iters + 1)]

pct = 0.75
n_zero = int(k * pct)

history = []

# Training
for it in range(iters):

    # Find misclassified observations
    margins = y * (X @ weights + bias)
    indexes = np.where(margins < 0)[0]

    history.append(len(indexes))

    if len(indexes) == 0:
        print(f"Converged after {it} iterations!")
        break

    # Select random misclassified observation
    idx = rng.choice(indexes)

    # SGD update
    weights += a[it] * y[idx] * X[idx]
    bias += a[it] * y[idx]

    # Prune smallest 75% of absolute weights
    sorted_indices = np.argsort(np.abs(weights))
    weights[sorted_indices[:n_zero]] = 0

else:
    print("Maximum iterations reached.")

# Evaluate
predictions = np.where(X @ weights + bias >= 0, 1, -1)

print(pd.crosstab(
    pd.Series(y, name="Actual"),
    pd.Series(predictions, name="Predicted")
))

print("Nonzero weights:", np.count_nonzero(weights))
print("Zero weights:", np.sum(weights == 0))

Maximum iterations reached.
Predicted  -1   1
Actual           
-1.00000   10   5
1.00000     1  30
Nonzero weights: 101
Zero weights: 300
